In [ ]:
# Imports
import json
import uuid
from datetime import datetime, timezone

import requests

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

In [ ]:
# Configuration
VAULT_URL = "https://mbta-api-key.vault.azure.net/"
SECRET_NAME = "mbta-api-key"
ROUTE_FILTER = "Red"

In [ ]:
# Get the API key
def get_api_key():
    return notebookutils.credentials.getSecret(VAULT_URL, SECRET_NAME)

In [ ]:
# Fetch routes
def fetch_routes(route_id):
    resp = requests.get(
        "https://api-v3.mbta.com/routes",
        params={"filter[id]": route_id},
        headers={"x-api-key": mbta_api_key}
    )
    resp.raise_for_status()
    return resp.json()

In [ ]:
# Fetch stops
def fetch_stops(route_id):
    resp = requests.get(
        "https://api-v3.mbta.com/stops",
        params={
            "filter[route]": route_id,
            "include": "child_stops"
        },
        headers={"x-api-key": mbta_api_key},
        timeout=30
    )

    resp.raise_for_status()

    api_response = resp.json()

    # Parent stations + included child stops
    all_stops = (
        api_response.get("data", [])
        + api_response.get("included", [])
    )

    # Keep only stop resources and remove duplicates
    unique_stops = {
        s["id"]: s
        for s in all_stops
        if s.get("type") == "stop"
    }

    return {"data": list(unique_stops.values())}

In [ ]:
# Fetch trips
def fetch_trips(route_id):
    resp = requests.get(
        "https://api-v3.mbta.com/trips",
        params={"filter[route]": route_id},
        headers={"x-api-key": mbta_api_key}
    )
    resp.raise_for_status()
    return resp.json()

In [ ]:
# Routes schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

def get_routes_schema():
    return StructType([
        StructField("route_id", StringType(), True),
        StructField("short_name", StringType(), True),
        StructField("long_name", StringType(), True),
        StructField("description", StringType(), True),
        StructField("color", StringType(), True),
        StructField("text_color", StringType(), True),
        StructField("type", IntegerType(), True),
        StructField("sort_order", IntegerType(), True),
        StructField("direction_names", StringType(), True),
        StructField("direction_destinations", StringType(), True),
        StructField("raw_json", StringType(), True),
        StructField("ingested_at", TimestampType(), True),
    ])

In [ ]:
# Flatten routes
def flatten_routes(api_response):
    ingested_at = datetime.now(timezone.utc)
    rows = []
    for r in api_response.get("data", []):
        a = r["attributes"]
        rows.append({
            "route_id": r["id"],
            "short_name": a.get("short_name"),
            "long_name": a.get("long_name"),
            "description": a.get("description"),
            "color": a.get("color"),
            "text_color": a.get("text_color"),
            "type": a.get("type"),
            "sort_order": a.get("sort_order"),
            "direction_names": json.dumps(a.get("direction_names")),
            "direction_destinations": json.dumps(a.get("direction_destinations")),
            "raw_json": json.dumps(r),
            "ingested_at": ingested_at,
        })
    return rows

In [ ]:
# Stops schema
def get_stops_schema():
    return StructType([
        StructField("stop_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("parent_station_id", StringType()),
        StructField("description", StringType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("municipality", StringType(), True),
        StructField("location_type", IntegerType(), True),
        StructField("platform_code", StringType(), True),
        StructField("platform_name", StringType(), True),
        StructField("wheelchair_boarding", IntegerType(), True),
        StructField("raw_json", StringType(), True),
        StructField("ingested_at", TimestampType(), True),
    ])

In [ ]:
# Flatten stops
def flatten_stops(api_response):
    ingested_at = datetime.now(timezone.utc)
    rows = []

    for s in api_response.get("data", []):
        a = s.get("attributes", {})

        parent_station = (
            s.get("relationships", {})
             .get("parent_station", {})
             .get("data") or {}
        )

        rows.append({
            "stop_id": s["id"],
            "name": a.get("name"),
            "parent_station_id": parent_station.get("id"),
            "description": a.get("description"),
            "latitude": a.get("latitude"),
            "longitude": a.get("longitude"),
            "municipality": a.get("municipality"),
            "location_type": a.get("location_type"),
            "platform_code": a.get("platform_code"),
            "platform_name": a.get("platform_name"),
            "wheelchair_boarding": a.get("wheelchair_boarding"),
            "raw_json": json.dumps(s),
            "ingested_at": ingested_at,
        })

    return rows

In [ ]:
# Trips schema
def get_trips_schema():
    return StructType([
        StructField("trip_id", StringType(), True),
        StructField("route_id", StringType(), True),
        StructField("headsign", StringType(), True),
        StructField("name", StringType(), True),
        StructField("direction_id", IntegerType(), True),
        StructField("block_id", StringType(), True),
        StructField("wheelchair_accessible", IntegerType(), True),
        StructField("bikes_allowed", IntegerType(), True),
        StructField("raw_json", StringType(), True),
        StructField("ingested_at", TimestampType(), True),
    ])

In [ ]:
# Flatten trips
def flatten_trips(api_response):
    ingested_at = datetime.now(timezone.utc)
    rows = []
    for t in api_response.get("data", []):
        a = t["attributes"]
        rel = t["relationships"]
        route_ref = rel.get("route", {}).get("data")
        rows.append({
            "trip_id": t["id"],
            "route_id": route_ref["id"] if route_ref else None,
            "headsign": a.get("headsign"),
            "name": a.get("name"),
            "direction_id": a.get("direction_id"),
            "block_id": a.get("block_id"),
            "wheelchair_accessible": a.get("wheelchair_accessible"),
            "bikes_allowed": a.get("bikes_allowed"),
            "raw_json": json.dumps(t),
            "ingested_at": ingested_at,
        })
    return rows

In [ ]:
# Write to Bronze (overwrite)
def write_to_bronze(rows, schema, table_name):
    df = spark.createDataFrame(rows, schema=schema)
    df.write.mode("overwrite").saveAsTable(table_name)
    return len(rows)

In [ ]:
# Audit log schema
def get_pipeline_runs_schema():
    return StructType([
        StructField("run_id", StringType(), True),
        StructField("pipeline_run_id", StringType(), True),
        StructField("stage_name", StringType(), True),
        StructField("status", StringType(), True),
        StructField("rows_affected", IntegerType(), True),
        StructField("started_at", TimestampType(), True),
        StructField("completed_at", TimestampType(), True),
        StructField("error_message", StringType(), True),
    ])

In [ ]:
# Log a run
def log_run(stage_name, status, rows_affected, started_at, error_message=None):
    log_row = [{
        "run_id": str(uuid.uuid4()),
        "pipeline_run_id": None,
        "stage_name": stage_name,
        "status": status,
        "rows_affected": rows_affected,
        "started_at": started_at,
        "completed_at": datetime.now(timezone.utc),
        "error_message": error_message
    }]
    df = spark.createDataFrame(log_row, schema=get_pipeline_runs_schema())
    df.write.mode("append").synapsesql("WH_MBTA.control.pipeline_runs")

In [ ]:
# Run one reference source end-to-end
def run_reference_ingest(stage_name, fetch_fn, flatten_fn, schema, table_name):
    started_at = datetime.now(timezone.utc)
    try:
        api_response = fetch_fn(ROUTE_FILTER)
        rows = flatten_fn(api_response)
        row_count = write_to_bronze(rows, schema, table_name)
        log_run(stage_name, "SUCCESS", row_count, started_at)
        print(f"{stage_name}: {row_count} rows")
    except Exception as e:
        log_run(stage_name, "FAILED", 0, started_at, str(e))
        print(f"{stage_name} failed: {e}")
        raise

In [ ]:
# Get the API key
mbta_api_key = get_api_key()
print("Key retrieved successfully:", mbta_api_key is not None)

In [ ]:
# Run all three reference sources
run_reference_ingest("bronze_routes_ingest", fetch_routes, flatten_routes, get_routes_schema(), "bronze.routes_raw")
run_reference_ingest("bronze_stops_ingest", fetch_stops, flatten_stops, get_stops_schema(), "bronze.stops_raw")
run_reference_ingest("bronze_trips_ingest", fetch_trips, flatten_trips, get_trips_schema(), "bronze.trips_raw")